In [1]:
from pyspark.sql import SparkSession, Row
from pyspark.sql.functions import *

In [2]:
spark = (SparkSession
        .builder
        .appName("Spark Higher Order Functions")
        .getOrCreate())

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/10 15:41:19 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


First we will setup some data to demonstrate higher order functions.

In [ ]:
from pyspark.sql.types import *
schema = StructType([StructField("celsius", ArrayType(IntegerType()))])

In [ ]:
t_list = [[35, 36, 32, 30, 40, 42, 38]], [[31, 32, 34, 55, 56]]
t_c = spark.createDataFrame(t_list, schema)
t_c.createOrReplaceTempView("tC")

In [ ]:
# Show the DataFrame
t_c.show()

## transform()
`transform(array<T>, function<T, U>): array<U>`

The transform() function produces an array by applying a function to each element
of the input array (similar to a `map()` function):


In [ ]:
#Calculate Fahrenheit from Celsius for an array of temperatures

spark.sql("""
SELECT celsius,
    transform(celsius, t -> ((t * 9) div 5) + 32) as fahrenheit
    FROM tC
""").show()


## filter()
`filter(array<T>, function<T, Boolean>): array<T>`

The `filter()` function produces an array consisting of only the elements of the input
array for which the Boolean function is true:


In [ ]:
spark.sql("""
    SELECT celsius,
    filter(celsius, t -> t > 38) as high
    FROM tC
""").show()


## exists()
`exists(array<T>, function<T, V, Boolean>): Boolean`

The `exists()` function returns true if the Boolean function holds for any element in
the input array:



In [ ]:
spark.sql("""
    SELECT celsius,
    exists(celsius, t -> t = 38) as threshold
    FROM tC
""").show()

## reduce()
`reduce(array<T>, B, function<B, T, B>, function<B, R>)`

The `reduce()` function reduces the elements of the array to a single value by merging
the elements into a buffer B using `function<B, T, B>` and applying a finishing
`function<B, R>` on the final buffer:


In [ ]:
spark.sql("""
    SELECT celsius,
    reduce(
    celsius,
    0,
    (t, acc) -> t + acc,
    acc -> (acc div size(celsius) * 9 div 5) + 32
    ) as avgFahrenheit
    FROM tC
""").show()

Let's set up some data for more higher order functions.

In [ ]:
from pyspark.sql.functions import expr
tripdelaysFilePath = "../spark-sql-python/flights/data/departuredelays.csv"
airportsnaFilePath = "../spark-sql-python/flights/data/airport-codes-na.txt"

In [ ]:
# Obtain airport data set
airportsna = (spark.read
.format("csv")
.options(header="true", inferSchema="true", sep="\t")
.load(airportsnaFilePath))

In [ ]:
airportsna.createOrReplaceTempView("airports_na")

In [ ]:
# Obtain departure delays data set
departureDelays = (spark.read
.format("csv")
.options(header="true")
.load(tripdelaysFilePath))

In [ ]:
departureDelays = (departureDelays
.withColumn("delay", expr("CAST(delay as INT) as delay"))
.withColumn("distance", expr("CAST(distance as INT) as distance")))

departureDelays.createOrReplaceTempView("departureDelays")

In [ ]:
table1 = (departureDelays
.filter(expr("""origin == 'SEA' and destination == 'SFO' and
date like '01010%' and delay > 0""")))

table1.createOrReplaceTempView("table1")

In [ ]:
spark.sql("SELECT * FROM airports_na LIMIT 10").show()

In [ ]:
spark.sql("SELECT * FROM departureDelays LIMIT 10").show()

In [ ]:
spark.sql("SELECT * FROM table1").show()

## Union

A common pattern within Apache Spark is to union two different DataFrames with
the same schema together. This can be achieved using the union() method:


In [ ]:
# Union two tables
table2 = departureDelays.union(table1)
table2.createOrReplaceTempView("table2")
# Show the union (filtering for SEA and SFO in a specific time range)
table2.filter(expr("""origin == 'SEA' AND destination == 'SFO'
AND date LIKE '01010%' AND delay > 0""")).show()


In [ ]:
spark.sql("""
SELECT *
FROM table2
WHERE origin = 'SEA'
AND destination = 'SFO'
AND date LIKE '01010%'
AND delay > 0
""").show()


## Joins
A common DataFrame operation is to join two DataFrames (or tables) together. By
default, a Spark SQL join is an inner join, with the options being inner, cross,
outer, full, full_outer, left, left_outer, right, right_outer, left_semi, and
left_anti. 

The following code sample performs the default of an inner join between the `airportsna` and `table1` DataFrames:


In [ ]:
# Join departure delays data (foo) with airport info
table1.join(
airportsna,
airportsna.IATA == table1.origin
).select("City", "State", "date", "delay", "distance", "destination").show()

## Windowing
A window function uses values from the rows in a window (a range of input rows) to
return a set of values, typically in the form of another row. With window functions, it is possible to operate on a group of rows while still returning a single value for every input row. In this section, we will show how to use the dense_rank() window function; there are many other functions, as noted in the table below:


| Function Type | SQL | DataFrame API |
| :--- | :--- | :--- |
| Ranking functions | rank() | rank() |
| | dense_rank() | denseRank() |
| | percent_rank() | percentRank() |
| | ntile() | ntile() |
| | row_number() | rowNumber() |
| Analytic functions | cume_dist() | cumeDist() |
| | first_value() | firstValue() |
| | last_value() | lastValue() |
| | lag() | lag() |
| | lead() | lead() |



Let’s start with a review of the TotalDelays (calculated by sum(Delay)) experienced
by flights originating from Seattle (SEA), San Francisco (SFO), and New York City
(JFK) and going to a specific set of destination locations, as noted in the following
query:


In [ ]:
spark.sql("""DROP TABLE IF EXISTS departureDelaysWindow;""")


In [ ]:
spark.sql("""
CREATE TABLE departureDelaysWindowNew AS
SELECT origin, destination, SUM(delay) AS TotalDelays
FROM departureDelays
WHERE origin IN ('SEA', 'SFO', 'JFK')
AND destination IN ('SEA', 'SFO', 'JFK', 'DEN', 'ORD', 'LAX', 'ATL')
GROUP BY origin, destination;
""")

In [ ]:
spark.sql("""SELECT * FROM departureDelaysWindowNew""").show()

What if for each of these origin airports we wanted to find the three destinations that experienced the most delays? We could achieve this by running three different queries for each origin and then unioning the results together, like this:
```
-- In SQL
SELECT origin, destination, SUM(TotalDelays) AS TotalDelays
FROM departureDelaysWindow
WHERE origin = '[ORIGIN]'
GROUP BY origin, destination
ORDER BY SUM(TotalDelays) DESC
LIMIT 3
```

But a better approach would be to use a window function like `dense_rank()` to perform the following calculation:



In [ ]:
spark.sql("""
SELECT origin, destination, TotalDelays, rank
FROM (
SELECT origin, destination, TotalDelays, dense_rank()
OVER (PARTITION BY origin ORDER BY TotalDelays DESC) as rank
FROM departureDelaysWindowNew
) t
WHERE rank <= 3
""").show()

By using the `dense_rank()` window function, we can quickly ascertain that the destinations with the worst delays for the three origin cities were:

 * Seattle (SEA): San Francisco (SFO), Denver (DEN), and Chicago (ORD)
 * San Francisco (SFO): Los Angeles (LAX), Chicago (ORD), and New York (JFK)
 * New York (JFK): Los Angeles (LAX), San Francisco (SFO), and Atlanta (ATL)
   
It’s important to note that each window grouping needs to fit in a single executor and
will get composed into a single partition during execution. Therefore, we need to
ensure that our queries are not unbounded (i.e., limit the size of your window).

Notes:
-----

`row_number()`: Assigns a strictly unique, sequential number regardless of ties.	1, 2, 3, 4

`rank()`: Assigns the same rank to ties, but skips ranks to account for the duplicates.	1, 2, 2, 4

`dense_rank()`: Assigns the same rank to ties and never skips any numbers.	1, 2, 2, 3



## Modifications
Another common operation is to perform modifications to the DataFrame. While
DataFrames themselves are immutable, we can modify them through operations that
create new, different DataFrames, with different columns, for example. (Recall from
earlier chapters that the underlying RDDs are immutable—i.e., they cannot be
changed—to ensure there is data lineage for Spark operations.) 

Let’s start with our previous small DataFrame example:


In [ ]:
table1.show()

### Adding Columns

In [ ]:
from pyspark.sql.functions import expr
table3 = (table1.withColumn(
"status",
expr("CASE WHEN delay <= 10 THEN 'On-time' ELSE 'Delayed' END")
))


In [ ]:
table3.show()

### Dropping columns
To drop a column, use the drop() method. For example, let’s remove the delay column as we now have a status column, added in the previous section:


In [ ]:
table4 = table3.drop("delay")
table4.show()


### Renaming Columns

In [ ]:

table5 = table4.withColumnRenamed("status", "flight_status")
table5.show()


## Pivoting
When working with your data, sometimes you will need to swap the columns for the
rows—i.e., pivot your data.


In [ ]:
spark.sql("""
SELECT destination, CAST(SUBSTRING(date, 0, 2) AS int) AS month, delay
FROM departureDelays
WHERE origin = 'SEA'
""").show()

Pivoting allows you to place names in the month column (instead of 1 and 2 you can
show Jan and Feb, respectively) as well as perform aggregate calculations (in this case average and max) on the delays by destination and month:


In [ ]:
spark.sql("""
    SELECT * FROM (
    SELECT destination, CAST(SUBSTRING(date, 0, 2) AS int) AS month, delay
    FROM departureDelays WHERE origin = 'SEA'
    )
    PIVOT (
    CAST(AVG(delay) AS DECIMAL(4, 2)) AS AvgDelay, MAX(delay) AS MaxDelay
    FOR month IN (1 JAN, 2 FEB)
    )
    ORDER BY destination
""").show()

In [ ]:
spark.stop()